In [ ]:
# from LT_dataset.CIFAR_LT import IMBALANCECIFAR100
# dataset = IMBALANCECIFAR100(root='Path/To/data', imb_type='exp', imb_factor=0.01, rand_number=0, train=True, download=False)
# from LT_dataset.ImageNet_LT import ImageNet_LT
# dataset = ImageNet_LT('Path/To/datasets/ImageNet_LT')
# from LT_dataset.Places_LT import Places_LT

# dataset = Places_LT('Path/To/datasets/Places_LT')

from LT_dataset.inat2018 import iNaturalist2018
dataset = iNaturalist2018("Path/To/datasets/Places_LT")

In [6]:
import os
import json
import time
from openai import OpenAI

all_classes = dataset.classes
all_class_set = set(all_classes)       

client = OpenAI(
    api_key="sk-XXX",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

BATCH_SIZE = 10
SLEEP = 1.5
OUTPUT_PATH = "inat2018.json"
results = {}

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def clean_batch_result(batch_result, all_class_set):
    cleaned = {}
    for key, values in batch_result.items():
        if key not in all_class_set:
            continue
        valid_values = [v for v in values if v in all_class_set and v != key]
        cleaned[key] = valid_values
    return cleaned

def save_results(results, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)


In [ ]:
token_count = 0
for i, batch in enumerate(chunks(all_classes, BATCH_SIZE)):
    print(f"\n🔹 Batch {i+1}: {len(batch)} classes")

    prompt = f"""
You are given a complete list of all_classes:
{all_classes}

Task: Find similar classes for {batch}, using items from the complete list above.

Requirements:
- Return a valid JSON object only, strictly matching this format:
{{
  "class1": ["similar1", "similar2", ...],
  "class2": ["similar1", "similar2", ...],
  ...
}}
- You MUST choose only from the provided complete list.
- Include conceptually or visually related ones (e.g., same category, color, or function).
- Similarity should be based on **semantic meaning** (synonyms, same category, or visually related).
- Do **NOT** include any explanation or text outside the JSON.
"""

    while True:
        try:
            completion = client.chat.completions.create(
                model="qwen-plus",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0,
            )

            content = completion.choices[0].message.content.strip()

            total_tokens = completion.usage.total_tokens
            token_count += total_tokens
            batch_result = json.loads(content)

            batch_result = clean_batch_result(batch_result, all_class_set)

            results.update(batch_result)

            save_results(results, OUTPUT_PATH)

            print(f"Batch {i+1} done, saved to {OUTPUT_PATH}. spend token {total_tokens}  all_spend {token_count}")
            break

        except Exception as e:
            print(f"Error parsing batch {i+1}, retrying... ({e})")
            time.sleep(SLEEP)

    time.sleep(SLEEP)

print(f"All done! Final results saved to {OUTPUT_PATH}")

# cifar100 spend tokens: 8390
# image_net spes`end tokens:539602
# places spend tokens: 85622
# inat2018 spend tokens: 57292254


🔹 Batch 1: 10 classes
Batch 1 done, saved to inat2018.json. spend token 70377  all_spend 70377

🔹 Batch 2: 10 classes
Batch 2 done, saved to inat2018.json. spend token 69998  all_spend 140375

🔹 Batch 3: 10 classes
Batch 3 done, saved to inat2018.json. spend token 69982  all_spend 210357

🔹 Batch 4: 10 classes
Batch 4 done, saved to inat2018.json. spend token 69957  all_spend 280314

🔹 Batch 5: 10 classes
Batch 5 done, saved to inat2018.json. spend token 70047  all_spend 350361

🔹 Batch 6: 10 classes
Batch 6 done, saved to inat2018.json. spend token 70200  all_spend 420561

🔹 Batch 7: 10 classes
Batch 7 done, saved to inat2018.json. spend token 70364  all_spend 490925

🔹 Batch 8: 10 classes
Batch 8 done, saved to inat2018.json. spend token 70353  all_spend 561278

🔹 Batch 9: 10 classes
Batch 9 done, saved to inat2018.json. spend token 70246  all_spend 631524

🔹 Batch 10: 10 classes
Batch 10 done, saved to inat2018.json. spend token 70103  all_spend 701627

🔹 Batch 11: 10 classes
Batch